In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

df = pd.read_csv('ForeignGifts_edu.csv',low_memory=False)

In [2]:
#Institution Status as public private

public_universities = {
    "Arizona State University", "Auburn University Montgomery", "Ball State University",
    "Boise State University", "CUNY Bernard M. Baruch College", "California State Polytechnic University, Pomona",
    "California State University - Sacramento", "California State University Maritime Academy",
    "California State University, Bakersfield", "California State University, Chico",
    "California State University, Dominguez Hills", "California State University, East Bay",
    "California State University, Fresno", "California State University, Fullerton",
    "California State University, Long Beach", "California State University, Los Angeles",
    "California State University, Northridge", "California State University, San Bernardino",
    "California State University, San Marcos", "Central Michigan University", "Clemson University",
    "Cleveland State University", "College of William & Mary", "Colorado State University",
    "Des Moines Area Community College", "Eastern Washington University", "Florida International University",
    "Frostburg State University", "George Mason University", "Georgia Institute of Technology",
    "Georgia Southern University", "Houston Community College", "Indiana State University",
    "Indiana University - Bloomington", "Indiana University - Purdue University Indianapolis",
    "Indiana University of Pennsylvania", "Iowa State University of Science & Technology",
    "Jacksonville State University", "Kansas State University", "Kean University",
    "Kennesaw State University", "Kent State University", "Lamar University", "Metropolitan State University",
    "Miami University", "Michigan State University", "Michigan Technological University",
    "Middle Tennessee State University", "Midwestern State University", "Missouri Southern State University",
    "Missouri State University", "Missouri University of Science and Technology", "Murray State University",
    "New Jersey Institute of Technology", "New Mexico State University",
    "North Carolina Agricultural & Technical State University", "North Carolina State University",
    "North Dakota State University - Fargo", "Northeastern State University", "Northern Arizona University",
    "Northern Illinois University", "Ohio State University (The)", "Ohio University", "Oklahoma State University",
    "Oregon Health & Science University", "Oregon State University", "Pennsylvania State University (The)",
    "Purdue University", "Rutgers, the State University of New Jersey", "San Diego State University",
    "San Francisco State University", "San Jose State University", "Texas A&M University",
    "Texas Tech University", "Towson University", "Troy University", "University of Akron (The)",
    "University of Alabama", "University of Alabama at Birmingham", "University of Arizona (The)",
    "University of Arkansas", "University of California, Berkeley", "University of California, Davis",
    "University of California, Irvine", "University of California, Los Angeles",
    "University of California, Merced", "University of California, Riverside",
    "University of California, San Diego", "University of California, San Francisco",
    "University of California, Santa Barbara", "University of California, Santa Cruz",
    "University of Central Florida", "University of Central Oklahoma", "University of Cincinnati",
    "University of Colorado Boulder", "University of Colorado Colorado Springs", "University of Colorado Denver",
    "University of Connecticut", "University of Florida", "University of Georgia",
    "University of Hawaii at Hilo", "University of Hawaii at Manoa", "University of Houston",
    "University of Idaho", "University of Illinois at Chicago", "University of Illinois at Urbana-Champaign",
    "University of Iowa", "University of Kansas", "University of Kentucky", "University of Louisville",
    "University of Maine", "University of Maryland, Baltimore", "University of Maryland, College Park",
    "University of Massachusetts - Amherst", "University of Michigan - Ann Arbor",
    "University of Minnesota - Twin Cities", "University of Mississippi", "University of Missouri - Columbia",
    "University of Missouri - Kansas City", "University of Missouri - Saint Louis", "University of Nebraska",
    "University of Nebraska Medical Center", "University of Nebraska at Omaha", "University of New Hampshire",
    "University of North Carolina - Chapel Hill", "University of North Carolina - Charlotte",
    "University of North Carolina -Greensboro", "University of North Texas",
    "University of North Texas Health Science Center at Fort Worth", "University of Northern Colorado",
    "University of Northern Iowa", "University of Oklahoma", "University of Oregon",
    "University of South Alabama", "University of South Florida",
    "University of Tennessee", "University of Tennessee Health Science Center",
    "University of Texas Health Science Center at Houston", "University of Texas Health Science Center at San Antonio",
    "University of Texas MD Anderson Cancer Center", "University of Texas Medical Branch at Galveston",
    "University of Texas Southwestern Medical Center (The)", "University of Texas at Arlington",
    "University of Texas at Austin", "University of Texas at Dallas", "University of Texas at San Antonio",
    "University of Toledo", "University of Utah", "University of Vermont and State Agricultural College",
    "University of Virginia", "University of Washington - Seattle", "University of Wisconsin - Madison",
    "University of Wisconsin - Platteville", "University of Wyoming", "Virginia Commonwealth University",
    "Virginia Polytechnic Institute & State University", "Washington State University", "Wayne State University",
    "West Virginia University", "Western Michigan University", "Western Oregon University",
    "Wichita State University", "Winthrop University", "Wright State University",
}

df["is_public"] = df["Institution Name"].isin(public_universities).astype(int)



In [3]:
#Giftor classification

import re

def classify_giftor(name):
    if pd.isna(name):
        return "Anonymous"
    n = str(name).strip()
    nl = n.lower()

    anon_kw = ["anonymous", "anon.", "anon donor", "anon #", "donor #"]
    if any(k in nl for k in anon_kw):
        return "Anonymous"
    if nl in ("individual", "individuals", "private individual"):
        return "Individual"

    gov_kw = ["embassy", "ministry", "ministerio", "ministere", "consulate", "government",
              "republic of", "cultural mission", "cultural office", "municipality",
              "federal ", "prefecture", "city of ", "united nations", "world health organization",
              "world bank", "european union", "european commission", "provincial", "province ",
              "state of ", "national council", "national research council", "consejo nacional",
              "conselho nacional", "national institutes of health", "national science foundation",
              "national natural science", "agency for", "anti doping agency",
              "organization for scientific", "organisation for scientific"]
    if any(k in nl for k in gov_kw):
        return "Government/International Body"

    academic_kw = ["university", "universit", "univ.", "univ ", " u of ", "college", "institute",
                   "institut ", "instituut", "ecole ", "école", "universidad", "school of",
                   "business school", "academy", "hospital", "medical center", "medical centre",
                   "health care", "polytechnic", "conservatory", "center for", "centre for",
                   "research center", "research centre", "laborator", "labratory", "national lab",
                   "publishing", " press", "society of", "geological society"]
    if any(k in nl for k in academic_kw):
        return "Academic/Research Institution"

    charity_kw = ["foundation", "fdn ", "fdn.", "stichting", "trust", "charity", "charitable",
                  "philanthrop", "fund", "nonprofit", "non-profit", "ngo", "rotary club",
                  "education mission", "scholar", "memorial"]
    if any(k in nl for k in charity_kw):
        return "Charitable Organization"

    corp_kw = [r"\bcorp\b", "corporation", r"\binc\b", r"\bllc\b", r"\bllp\b", r"\bltd\b",
               r"\bco\b", " company", " group", " holdings", "pharmaceutical", "pharma",
               "biologic", r"\bag\b", "gmbh", r"\bs\.a\.", r"\bsa\b", r"\bspa\b", r"\bplc\b",
               r"\bb\.v\.", r"\bbv\b", r"\bsas\b", r"\bsarl\b", r"\boy\b", r"\bab\b",
               "limited", "industries", "enterprises", " partners", " bank", " motor",
               "technolog", " systems", " solutions", "electronics", "biosciences",
               "biotech", "laboratories", "life sciences", "healthcare", "chemicals",
               "compania", "companhia", "compagnie", "aktiengesellschaft", "energy",
               "resources", "capital", "ventures", "consulting", "services",
               "international", "global", "worldwide", "kaisha", "kabushiki", r"\bkk\b",
               r"\basa\b", "plasma", "medical services", "diagnostics", "therapeutics",
               "biopharma", "genomics", "instruments", "manufacturing", "electric ",
               "petroleum", "chemical ", "investment", "asset management", "air lines",
               " airways", " airlines", " air$", "kunsthandel"]
    if any(re.search(k, nl) for k in corp_kw):
        return "Industry/Corporate"

    if re.match(r'^[a-z\-\']+,\s*[a-z]', nl) or re.match(r'^(mr|mrs|ms|dr|prof)\.?\s', nl) \
       or re.search(r'\bph\.?d\b', nl) or re.search(r'\besq\b', nl) or re.search(r'\bmd\b', nl):
        return "Individual"

    return "Individual"  # residual: mix of real individuals + unmatched orgs, see caveat below

df["giftor_category"] = df["Giftor Name"].apply(classify_giftor)

In [4]:
#Geograph location

# This a variable that codes gifter based on region

region_map = {
    'AFGHANISTAN': 'South Asia', 'AMERICAN SAMOA': 'East Asia & Pacific', 'ANGOLA': 'Sub-Saharan Africa',
    'ANTIGUA': 'Latin America & Caribbean', 'ARGENTINA': 'Latin America & Caribbean', 'ARMENIA': 'Europe & Central Asia',
    'AUSTRALIA': 'East Asia & Pacific', 'AUSTRIA': 'Europe & Central Asia', 'AZERBAIJAN': 'Europe & Central Asia',
    'BAHAMAS': 'Latin America & Caribbean', 'BAHRAIN': 'Middle East & North Africa', 'BANGLADESH': 'South Asia',
    'BARBADOS': 'Latin America & Caribbean', 'BELARUS': 'Europe & Central Asia', 'BELGIUM': 'Europe & Central Asia',
    'BERMUDA': 'North America', 'BOLIVIA': 'Latin America & Caribbean', 'BOSNIA AND HERZEGOVINA': 'Europe & Central Asia',
    'BOSNIA-HERCEGOVINA': 'Europe & Central Asia', 'BOTSWANA': 'Sub-Saharan Africa', 'BRAZIL': 'Latin America & Caribbean',
    'BRITISH WEST INDIES': 'Latin America & Caribbean', 'BULGARIA': 'Europe & Central Asia', 'BURKINA FASO': 'Sub-Saharan Africa',
    'BURMA': 'East Asia & Pacific', 'CAMBODIA': 'East Asia & Pacific', 'CANADA': 'North America',
    'CAYMAN ISLANDS (THE)': 'Latin America & Caribbean', 'CEYLON': 'South Asia', 'CHILE': 'Latin America & Caribbean',
    'CHINA': 'East Asia & Pacific', 'COLOMBIA': 'Latin America & Caribbean', 'COSTA RICA': 'Latin America & Caribbean',
    'CROATIA': 'Europe & Central Asia', 'CURAçAO': 'Latin America & Caribbean', 'CYPRUS': 'Europe & Central Asia',
    'CZECH REPUBLIC': 'Europe & Central Asia', "CôTE D'IVOIRE": 'Sub-Saharan Africa', 'DENMARK': 'Europe & Central Asia',
    'DOMINICAN REPUBLIC': 'Latin America & Caribbean', 'ECUADOR': 'Latin America & Caribbean', 'EGYPT': 'Middle East & North Africa',
    'EL SALVADOR': 'Latin America & Caribbean', 'ENGLAND': 'Europe & Central Asia', 'ERITREA': 'Sub-Saharan Africa',
    'ETHIOPIA': 'Sub-Saharan Africa', 'FINLAND': 'Europe & Central Asia', 'FRANCE': 'Europe & Central Asia',
    'GEORGIA': 'Europe & Central Asia', 'GERMANY': 'Europe & Central Asia', 'GHANA': 'Sub-Saharan Africa',
    'GREECE': 'Europe & Central Asia', 'GRENADA': 'Latin America & Caribbean', 'GUAM': 'East Asia & Pacific',
    'GUATEMALA': 'Latin America & Caribbean', 'GUERNSEY': 'Europe & Central Asia', 'HAITI': 'Latin America & Caribbean',
    'HONDURAS': 'Latin America & Caribbean', 'HONG KONG': 'East Asia & Pacific', 'HUNGARY': 'Europe & Central Asia',
    'ICELAND': 'Europe & Central Asia', 'INDIA': 'South Asia', 'INDONESIA': 'East Asia & Pacific',
    'IRAN': 'Middle East & North Africa', 'IRAQ': 'Middle East & North Africa', 'IRELAND': 'Europe & Central Asia',
    'ISLE OF MAN': 'Europe & Central Asia', 'ISRAEL': 'Middle East & North Africa', 'ITALY': 'Europe & Central Asia',
    'JAMAICA': 'Latin America & Caribbean', 'JAPAN': 'East Asia & Pacific', 'JERSEY': 'Europe & Central Asia',
    'JORDAN': 'Middle East & North Africa', 'KAZAKHSTAN': 'Europe & Central Asia', 'KENYA': 'Sub-Saharan Africa',
    'KOREA': 'East Asia & Pacific', 'KUWAIT': 'Middle East & North Africa', 'LATVIA': 'Europe & Central Asia',
    'LEBANON': 'Middle East & North Africa', 'LESOTHO': 'Sub-Saharan Africa', 'LIBERIA': 'Sub-Saharan Africa',
    'LIECHTENSTEIN': 'Europe & Central Asia', 'LITHUANIA': 'Europe & Central Asia', 'LUXEMBOURG': 'Europe & Central Asia',
    'MADAGASCAR': 'Sub-Saharan Africa', 'MALAWI': 'Sub-Saharan Africa', 'MALAYSIA': 'East Asia & Pacific',
    'MALI': 'Sub-Saharan Africa', 'MALTA': 'Europe & Central Asia', 'MARSHAL ISLANDS (THE)': 'East Asia & Pacific',
    'MAURITIUS': 'Sub-Saharan Africa', 'MEXICO': 'Latin America & Caribbean', 'MICRONESIA': 'East Asia & Pacific',
    'MONACO': 'Europe & Central Asia', 'MOROCCO': 'Middle East & North Africa', 'MOZAMBIQUE': 'Sub-Saharan Africa',
    'NAMIBIA': 'Sub-Saharan Africa', 'NEPAL': 'South Asia', 'NEW ZEALAND': 'East Asia & Pacific',
    'NIGERIA': 'Sub-Saharan Africa', 'NORTHERN IRELAND': 'Europe & Central Asia', 'NORTHERN MARIANA ISL': 'East Asia & Pacific',
    'NORWAY': 'Europe & Central Asia', 'OMAN': 'Middle East & North Africa', 'PAKISTAN': 'South Asia',
    'PALAU': 'East Asia & Pacific', 'PALESTINE, STATE OF': 'Middle East & North Africa', 'PANAMA': 'Latin America & Caribbean',
    'PAPUA NEW GUINEA': 'East Asia & Pacific', 'PERU': 'Latin America & Caribbean', 'PHILIPPINES': 'East Asia & Pacific',
    'POLAND': 'Europe & Central Asia', 'PORTUGAL': 'Europe & Central Asia', 'PUERTO RICO': 'Latin America & Caribbean',
    'QATAR': 'Middle East & North Africa', 'ROMANIA': 'Europe & Central Asia', 'RUSSIA': 'Europe & Central Asia',
    'RWANDA': 'Sub-Saharan Africa', 'SAUDI ARABIA': 'Middle East & North Africa', 'SCOTLAND': 'Europe & Central Asia',
    'SENEGAL': 'Sub-Saharan Africa', 'SERBIA': 'Europe & Central Asia', 'SINGAPORE': 'East Asia & Pacific',
    'SLOVAKIA': 'Europe & Central Asia', 'SLOVENIA': 'Europe & Central Asia', 'SOUTH AFRICA': 'Sub-Saharan Africa',
    'SOUTH KOREA': 'East Asia & Pacific', 'SPAIN': 'Europe & Central Asia', 'ST. KITTS-NEVIS': 'Latin America & Caribbean',
    'ST. LUCIA': 'Latin America & Caribbean', 'SWAZILAND': 'Sub-Saharan Africa', 'SWEDEN': 'Europe & Central Asia',
    'SWITZERLAND': 'Europe & Central Asia', 'SYRIAN ARAB REPUBLIC': 'Middle East & North Africa', 'TAIWAN': 'East Asia & Pacific',
    'TANZANIA': 'Sub-Saharan Africa', 'THAILAND': 'East Asia & Pacific', 'THE NETHERLANDS': 'Europe & Central Asia',
    'TOKELAU': 'East Asia & Pacific', 'TRINIDAD': 'Latin America & Caribbean', 'TUNISIA': 'Middle East & North Africa',
    'TURKEY': 'Europe & Central Asia', 'UGANDA': 'Sub-Saharan Africa', 'UKRAINE': 'Europe & Central Asia',
    'UNITED ARAB EMIRATES': 'Middle East & North Africa', 'UNITED NATIONS': 'Other / International',
    'URUGUAY': 'Latin America & Caribbean', 'USA': 'North America', 'VENEZUELA': 'Latin America & Caribbean',
    'VIETNAM': 'East Asia & Pacific', 'VIRGIN ISLANDS (BRITISH)': 'Latin America & Caribbean', 'WALES': 'Europe & Central Asia',
    'YEMEN': 'Middle East & North Africa', 'ZAMBIA': 'Sub-Saharan Africa', 'ZIMBABWE': 'Sub-Saharan Africa',
}

df['region'] = df['Country of Giftor'].map(region_map)

In [5]:
#Public ivies classification 
#Public ivy classification
# Tier 1: Richard Moll's original 1985 "Public Ivies" (where the term originated).
# The UC system was named as a single entry covering multiple campuses.
public_ivy_core = {
    "College of William & Mary",
    "Miami University",
    "University of California, Berkeley",
    "University of California, Los Angeles",
    "University of California, San Diego",
    "University of California, Irvine",
    "University of California, Davis",
    "University of California, Santa Barbara",
    "University of California, Santa Cruz",
    "University of California, Riverside",
    "University of Michigan - Ann Arbor",
    "University of North Carolina - Chapel Hill",
    "University of Texas at Austin",
    "University of Vermont and State Agricultural College",
    "University of Virginia",
}

# Tier 2: Howard & Matthew Greene's 2001 update, "The Public Ivies" —
# a superset that keeps everything above and adds more flagship public schools.
public_ivy_expanded = public_ivy_core | {
    "Rutgers, the State University of New Jersey",
    "University of Connecticut",
    "University of Delaware",
    "University of Maryland, College Park",
    "University of Arizona (The)",
    "University of Colorado Boulder",
    "University of Washington - Seattle",
    "Indiana University - Bloomington",
    "Michigan State University",
    "Ohio State University (The)",
    "University of Illinois at Urbana-Champaign",
    "University of Iowa",
    "University of Minnesota - Twin Cities",
    "University of Wisconsin - Madison",
    "University of Florida",
    "University of Georgia",
}

df["public_ivy_core"] = df["Institution Name"].isin(public_ivy_core).astype(int)
df["public_ivy_expanded"] = df["Institution Name"].isin(public_ivy_expanded).astype(int)

In [8]:
df.to_csv("ForeignGifts_dummyCoded.csv")